# Visualize Awinda (per-joint): Model ID vs OpenSim ID

Interactive plots for cached **single-joint** awinda results from `process_awinda_per_joint.ipynb`.

- **Caches**: `analysis/cache/process_awinda_per_joint_{hip,knee,ankle}.npz`
- **Sync**: Awinda vs Vicon IK angle xcorr (from process notebook); 15 s post-sync trim on ID metrics
- **IK QC**: synced Vicon vs Awinda IK for the selected joint (R/L)
- **ID QC**: Model ID vs OpenSim for the selected joint
- **Views**: sync diagnostics, per-trial ID/IK explorer
- **Paper outputs**: tables & figures under `analysis/paper_outputs/awinda_per_joint/{joint}/`

Run `process_awinda_per_joint.ipynb` first if caches are missing or stale.


In [ ]:
import io
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings('ignore', message='.*NumPy.*')

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
CACHE_DIR = PROJECT_ROOT / 'analysis' / 'cache'
SUBJECT_CSV_DIR = PROJECT_ROOT / 'analysis' / 'cache' / 'visualize_awinda_per_joint_by_subject'

JOINT_CACHE_PATHS = {
    'hip': CACHE_DIR / 'process_awinda_per_joint_hip.npz',
    'knee': CACHE_DIR / 'process_awinda_per_joint_knee.npz',
    'ankle': CACHE_DIR / 'process_awinda_per_joint_ankle.npz',
}

DEFAULT_PREVIEW_FRAMES = 500

available_joints = [j for j, p in JOINT_CACHE_PATHS.items() if p.is_file()]
missing_joints = [j for j, p in JOINT_CACHE_PATHS.items() if not p.is_file()]
print('Available joint caches:', available_joints)
if missing_joints:
    print('Missing joint caches (run process_awinda_per_joint.ipynb / train checkpoint):', missing_joints)
if not available_joints:
    raise FileNotFoundError('No process_awinda_per_joint_*.npz caches found.')


def _safe_subject_name(subject: str) -> str:
    return subject.replace(' ', '_')


def _subject_from_trial(trial_key: str) -> str:
    return trial_key.split('::', 1)[0]


def save_awinda_results_by_subject(
    joint: str,
    summary_df: pd.DataFrame,
    detail_df: pd.DataFrame,
    *,
    ik_summary_df: Optional[pd.DataFrame] = None,
    ik_detail_df: Optional[pd.DataFrame] = None,
) -> None:
    out_dir = SUBJECT_CSV_DIR / joint
    out_dir.mkdir(parents=True, exist_ok=True)

    def _attach_subject(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        if 'subject' not in out.columns:
            out['subject'] = out['trial'].map(_subject_from_trial)
        return out

    summary = _attach_subject(summary_df)
    detail = _attach_subject(detail_df)
    subjects = sorted(summary['subject'].unique())
    for subject in subjects:
        safe = _safe_subject_name(subject)
        sub_summary = summary.loc[summary['subject'] == subject].drop(columns=['subject'], errors='ignore')
        sub_detail = detail.loc[detail['subject'] == subject].drop(columns=['subject'], errors='ignore')
        sub_summary.to_csv(out_dir / f'{safe}_id_summary.csv', index=False)
        sub_detail.to_csv(out_dir / f'{safe}_id_detail.csv', index=False)

        if ik_summary_df is not None and not ik_summary_df.empty:
            ik_summary = _attach_subject(ik_summary_df)
            ik_detail = _attach_subject(ik_detail_df)
            ik_summary.loc[ik_summary['subject'] == subject].drop(columns=['subject'], errors='ignore').to_csv(
                out_dir / f'{safe}_ik_summary.csv', index=False)
            ik_detail.loc[ik_detail['subject'] == subject].drop(columns=['subject'], errors='ignore').to_csv(
                out_dir / f'{safe}_ik_detail.csv', index=False)

            combined = sub_summary.merge(
                ik_summary.loc[ik_summary['subject'] == subject, [
                    'trial', 'mean_rmse_deg', 'mean_r2', 'lag_samples',
                ]],
                on='trial',
                how='left',
            ).drop(columns=['subject'], errors='ignore')
            combined.to_csv(out_dir / f'{safe}_combined.csv', index=False)

    print(f'[{joint}] Saved per-subject CSVs for {len(subjects)} subjects -> {out_dir}')


In [ ]:
def _trial_key_to_prefix(trial_key: str) -> str:
    return trial_key.replace('::', '__')


def load_processed_awinda_per_joint_cache(path: Path):
    data = np.load(str(path), allow_pickle=True)
    joint = str(data['joint'].item()) if 'joint' in data.files else path.stem.split('_')[-1]
    channels = [str(c) for c in data['channels']]
    display_names = (
        [str(c) for c in data['display_names']]
        if 'display_names' in data.files else channels
    )
    checkpoint = str(data['checkpoint'].item()) if 'checkpoint' in data.files else ''
    preview_frames = int(data['preview_frames'].item()) if 'preview_frames' in data.files else DEFAULT_PREVIEW_FRAMES
    sync_method = str(data['sync_method'].item()) if 'sync_method' in data.files else 'angle_xcorr'
    transient_trim_sec = (
        float(data['transient_trim_sec'].item())
        if 'transient_trim_sec' in data.files else 15.0
    )

    trial_data = {}
    for trial_key in data['trial_keys']:
        key = str(trial_key)
        p = _trial_key_to_prefix(key)
        meta = data[f'{p}__meta']
        rmse = data[f'{p}__rmse_nmpkg']
        r2 = data[f'{p}__r2_nmpkg']
        metrics = [
            {'channel': channels[c], 'rmse_nmpkg': float(rmse[c]), 'r2_nmpkg': float(r2[c])}
            for c in range(len(channels))
        ]

        xcorr_score = float(meta[5])
        lag_clipped = bool(meta[6])

        entry = {
            'subject': str(meta[0]),
            'condition': str(meta[1]),
            'mass_kg': float(meta[2]),
            't': data[f'{p}__t'],
            'pred_nmpkg': data[f'{p}__pred_nmpkg'],
            'id_nmpkg': data[f'{p}__id_nmpkg'],
            'metrics': metrics,
            'lag_samples': int(meta[3]),
            'lag_seconds': float(meta[4]),
            'sync_method': sync_method,
            'xcorr_score': xcorr_score,
            'lag_clipped': lag_clipped,
            'joint': joint,
        }
        if f'{p}__sync_pred_pre' in data.files:
            entry['sync_debug'] = {
                'fs_hz': float(data[f'{p}__sync_fs_hz'].item()),
                'n_sync': int(data[f'{p}__sync_n'].item()),
                't_id': data[f'{p}__sync_t_id'],
                't_imu': data[f'{p}__sync_t_imu'],
                'pred_pre': data[f'{p}__sync_pred_pre'],
                'id_pre': data[f'{p}__sync_id_pre'],
                'pred_ankle': data[f'{p}__sync_pred_ankle'],
                'id_ankle': data[f'{p}__sync_id_ankle'],
                'lag': entry['lag_samples'],
                'sync_method': sync_method,
                'xcorr_score': entry['xcorr_score'],
            }
            if f'{p}__sync_awinda_hip_deg' in data.files:
                entry['sync_debug']['awinda_hip_deg'] = data[f'{p}__sync_awinda_hip_deg']
                entry['sync_debug']['vicon_hip_deg'] = data[f'{p}__sync_vicon_hip_deg']
        trial_data[key] = entry

    summary_df = (
        pd.read_json(io.StringIO(str(data['summary_df'].item())), orient='split')
        if 'summary_df' in data.files else pd.DataFrame()
    )
    detail_df = (
        pd.read_json(io.StringIO(str(data['detail_df'].item())), orient='split')
        if 'detail_df' in data.files else pd.DataFrame()
    )
    print(f'[{joint}] Loaded {len(trial_data)} trials from {path} (sync={sync_method})')
    return {
        'joint': joint,
        'channels': channels,
        'display_names': display_names,
        'checkpoint': checkpoint,
        'TRIAL_DATA': trial_data,
        'summary_df': summary_df,
        'detail_df': detail_df,
        'preview_frames': preview_frames,
        'transient_trim_sec': transient_trim_sec,
        'sync_method': sync_method,
    }


JOINT_BUNDLES = {j: load_processed_awinda_per_joint_cache(JOINT_CACHE_PATHS[j]) for j in available_joints}

for joint, bundle in JOINT_BUNDLES.items():
    detail_df = bundle['detail_df']
    if not detail_df.empty:
        print(f'\n[{joint}] Per-side mean across trials:')
        display(detail_df.groupby('channel')[['rmse_nmpkg', 'r2_nmpkg']].mean())
    if not bundle['summary_df'].empty:
        save_awinda_results_by_subject(joint, bundle['summary_df'], detail_df)


In [ ]:
import pickle
import sys
from scipy.signal import butter, sosfiltfilt

sys.path.insert(0, str(PROJECT_ROOT))
from dataset import IK_DOF_NAMES

PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
IMU_IK_ROOT = Path('/home/metamobility3/Jinwoo/mt_processed')
IMU_IK_METHOD = 'VQF'
ANGLE_CUTOFF_HZ = 6.0
FILTER_ORDER = 4

IK_MIN_R2 = 0.7
IK_MAX_RMSE_DEG = 15.0


def _parse_opensim_table(path: Path) -> pd.DataFrame:
    with open(path) as f:
        header_end = next(i for i, line in enumerate(f) if line.strip().lower() == 'endheader')
    return pd.read_csv(path, sep=r'\s+', skiprows=header_end + 1).set_index('time')


def _butter_lpf(x, fs_hz, cutoff_hz=ANGLE_CUTOFF_HZ, order=FILTER_ORDER):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * fs_hz
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(x) < 4:
        return x.copy()
    sos = butter(order, cutoff_hz / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, x)


def _lpf_mc(X, fs_hz):
    return np.column_stack([_butter_lpf(X[:, c], fs_hz) for c in range(X.shape[1])])


def _condition_to_pkl_stem(condition: str) -> str:
    cond, speed = condition.split('_', 1)
    return f'{speed}_{cond.lower()}'


def _build_awinda_ik_rad(imu_dict: dict) -> np.ndarray:
    n = len(next(iter(imu_dict.values())))
    pos_deg = np.zeros((n, len(IK_DOF_NAMES)), dtype=np.float64)
    key_map = {
        'hip_flexion_r': 'hip_flexion_r', 'knee_angle_r': 'knee_flexion_r', 'ankle_angle_r': 'ankle_flexion_r',
        'hip_flexion_l': 'hip_flexion_l', 'knee_angle_l': 'knee_flexion_l', 'ankle_angle_l': 'ankle_flexion_l',
    }
    sign_map = {'knee_angle_r': -1.0, 'knee_angle_l': -1.0}
    for ik_name, pkl_name in key_map.items():
        idx = IK_DOF_NAMES.index(ik_name)
        pos_deg[:, idx] = sign_map.get(ik_name, 1.0) * np.asarray(imu_dict[pkl_name], dtype=np.float64)
    return np.deg2rad(pos_deg)


def _build_vicon_ik_rad(ik_df: pd.DataFrame) -> np.ndarray:
    pos_deg = np.full((len(ik_df), len(IK_DOF_NAMES)), np.nan)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            pos_deg[:, j] = ik_df[name].to_numpy(dtype=np.float64)
    return np.deg2rad(pos_deg)


def _rmse_r2_deg(y_pred, y_ref):
    m = np.isfinite(y_pred) & np.isfinite(y_ref)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_ref[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    r2 = float(np.corrcoef(y_pred[m], y_ref[m])[0, 1] ** 2)
    return rmse, r2


def compute_ik_for_joint(bundle: dict):
    channels = bundle['channels']
    ik_channel_idx = [IK_DOF_NAMES.index(c) for c in channels]
    trim_sec = bundle['transient_trim_sec']
    TRIAL_DATA = bundle['TRIAL_DATA']

    ik_detail_rows = []
    ik_summary_rows = []
    ik_errors = []
    IK_TRIAL_DATA = {}

    for trial_key, d in sorted(TRIAL_DATA.items()):
        subject, cond = d['subject'], d['condition']
        lag = int(d['lag_samples'])
        try:
            pkl_path = IMU_IK_ROOT / subject / 'ik' / IMU_IK_METHOD / f'{_condition_to_pkl_stem(cond)}.pkl'
            vicon_path = PROCESSED_ROOT / subject / 'awinda' / 'ik' / f'{cond}_ik.mot'
            if not pkl_path.exists():
                raise FileNotFoundError(f'Missing Awinda IK pkl: {pkl_path}')
            if not vicon_path.exists():
                raise FileNotFoundError(f'Missing Vicon IK mot: {vicon_path}')

            awinda_rad = _build_awinda_ik_rad(pickle.load(open(pkl_path, 'rb')))
            vicon_df = _parse_opensim_table(vicon_path)
            t_vicon = vicon_df.index.to_numpy(dtype=np.float64)
            vicon_rad = _build_vicon_ik_rad(vicon_df)
            awinda_j = awinda_rad[:, ik_channel_idx]
            vicon_j = vicon_rad[:, ik_channel_idx]

            fs_hz = float(d['sync_debug']['fs_hz']) if 'sync_debug' in d else 100.0
            trim_n = int(round(trim_sec * fs_hz))
            start_awinda, start_vicon = max(lag, 0), max(-lag, 0)
            n_sync = min(len(awinda_j) - start_awinda, len(vicon_j) - start_vicon)
            if n_sync <= trim_n:
                raise RuntimeError(f'Not enough samples after {trim_sec}s trim: {n_sync}')

            awinda_sync = _lpf_mc(awinda_j[start_awinda:start_awinda + n_sync], fs_hz)
            vicon_sync = _lpf_mc(vicon_j[start_vicon:start_vicon + n_sync], fs_hz)
            awinda_eval = np.rad2deg(awinda_sync[trim_n:])
            vicon_eval = np.rad2deg(vicon_sync[trim_n:])
            t_eval = t_vicon[start_vicon + trim_n:start_vicon + n_sync]

            metrics = []
            for c, channel in enumerate(channels):
                rmse, r2 = _rmse_r2_deg(awinda_eval[:, c], vicon_eval[:, c])
                metrics.append({'channel': channel, 'rmse_deg': rmse, 'r2': r2})
                ik_detail_rows.append({
                    'trial': trial_key, 'subject': subject, 'condition': cond,
                    'channel': channel, 'rmse_deg': rmse, 'r2': r2,
                })

            mean_rmse = float(np.nanmean([m['rmse_deg'] for m in metrics]))
            mean_r2 = float(np.nanmean([m['r2'] for m in metrics]))
            IK_TRIAL_DATA[trial_key] = {
                'subject': subject,
                'condition': cond,
                't': t_eval,
                'awinda_deg': awinda_eval,
                'vicon_deg': vicon_eval,
                'metrics': metrics,
            }
            ik_summary_rows.append({
                'trial': trial_key,
                'subject': subject,
                'condition': cond,
                'n': len(awinda_eval),
                'mean_rmse_deg': mean_rmse,
                'mean_r2': mean_r2,
                'lag_samples': lag,
            })
        except Exception as exc:
            ik_errors.append(f'{trial_key}: {exc}')

    ik_summary_df = pd.DataFrame(ik_summary_rows)
    ik_detail_df = pd.DataFrame(ik_detail_rows)

    if ik_errors:
        print(f'[{bundle["joint"]}] IK comparison warnings:')
        for msg in ik_errors:
            print(f'  [WARN] {msg}')

    if ik_summary_df.empty:
        raise RuntimeError(f'[{bundle["joint"]}] No IK comparisons computed.')

    ik_flagged = ik_summary_df[
        (ik_summary_df['mean_r2'] < IK_MIN_R2) | (ik_summary_df['mean_rmse_deg'] > IK_MAX_RMSE_DEG)
    ].copy()
    ik_flagged['fail_r2'] = ik_flagged['mean_r2'] < IK_MIN_R2
    ik_flagged['fail_rmse'] = ik_flagged['mean_rmse_deg'] > IK_MAX_RMSE_DEG
    IK_BAD_TRIALS = set(ik_flagged['trial'])

    print(
        f'[{bundle["joint"]}] Synced Vicon IK vs Awinda IK ({trim_sec:.0f}s post-sync trim, 6 Hz zero-phase LPF)\n'
        f'  trials compared: {len(ik_summary_df)}\n'
        f'  QC: mean R2 < {IK_MIN_R2} or mean RMSE > {IK_MAX_RMSE_DEG} deg\n'
        f'  flagged: {len(ik_flagged)} / {len(ik_summary_df)}'
    )
    if not ik_flagged.empty:
        print('  Flagged IK trials by subject:')
        for subject in sorted(ik_flagged['subject'].unique()):
            sub = ik_flagged[ik_flagged['subject'] == subject].sort_values('condition')
            parts = []
            for _, row in sub.iterrows():
                reasons = []
                if row['fail_r2']:
                    reasons.append(f"R2={row['mean_r2']:.3f}")
                if row['fail_rmse']:
                    reasons.append(f"RMSE={row['mean_rmse_deg']:.2f} deg")
                parts.append(f"{row['condition']} ({', '.join(reasons)})")
            print(f'    {subject}: {", ".join(parts)}')

    print(f'\n[{bundle["joint"]}] Per-side mean across trials (IK vs Vicon):')
    display(ik_detail_df.groupby('channel')[['rmse_deg', 'r2']].mean())

    save_awinda_results_by_subject(
        bundle['joint'], bundle['summary_df'], bundle['detail_df'],
        ik_summary_df=ik_summary_df, ik_detail_df=ik_detail_df,
    )

    bundle['IK_TRIAL_DATA'] = IK_TRIAL_DATA
    bundle['ik_summary_df'] = ik_summary_df
    bundle['ik_detail_df'] = ik_detail_df
    bundle['IK_BAD_TRIALS'] = IK_BAD_TRIALS
    bundle['ik_flagged'] = ik_flagged
    return bundle


for joint in list(JOINT_BUNDLES):
    JOINT_BUNDLES[joint] = compute_ik_for_joint(JOINT_BUNDLES[joint])


In [ ]:
ID_MIN_R2 = 0.6
ID_MAX_RMSE = 0.24


def _annotate_id_qc(bundle: dict):
    TRIAL_DATA = bundle['TRIAL_DATA']
    summary_df = bundle['summary_df']
    IK_BAD_TRIALS = bundle['IK_BAD_TRIALS']

    if summary_df.empty:
        id_summary = pd.DataFrame([
            {
                'trial': k,
                'subject': d['subject'],
                'condition': d['condition'],
                'mean_rmse': float(np.nanmean([m['rmse_nmpkg'] for m in d['metrics']])),
                'mean_r2': float(np.nanmean([m['r2_nmpkg'] for m in d['metrics']])),
            }
            for k, d in TRIAL_DATA.items()
        ])
    else:
        id_summary = summary_df.copy()
        if 'subject' not in id_summary.columns:
            id_summary[['subject', 'condition']] = id_summary['trial'].str.split('::', n=1, expand=True)

    id_flagged = id_summary[
        (id_summary['mean_r2'] < ID_MIN_R2) | (id_summary['mean_rmse'] > ID_MAX_RMSE)
    ].copy()
    id_flagged['fail_r2'] = id_flagged['mean_r2'] < ID_MIN_R2
    id_flagged['fail_rmse'] = id_flagged['mean_rmse'] > ID_MAX_RMSE
    ID_BAD_TRIALS = set(id_flagged['trial'])

    id_flagged['ik_also_bad'] = id_flagged['trial'].isin(IK_BAD_TRIALS)
    id_flagged['bad_category'] = np.where(
        id_flagged['ik_also_bad'],
        'ID + IK bad',
        'ID only (IK OK)',
    )

    both_bad = ID_BAD_TRIALS & IK_BAD_TRIALS
    id_only_bad = ID_BAD_TRIALS - IK_BAD_TRIALS
    ik_only_bad = IK_BAD_TRIALS - ID_BAD_TRIALS

    print(
        f'[{bundle["joint"]}] ID QC: mean R2 < {ID_MIN_R2} or mean RMSE > {ID_MAX_RMSE} N·m/kg\n'
        f'  flagged: {len(id_flagged)} / {len(id_summary)}\n'
        f'  ID+IK bad: {len(both_bad)} | ID only: {len(id_only_bad)} | IK only: {len(ik_only_bad)}'
    )
    if not id_flagged.empty:
        display(
            id_flagged.sort_values(['subject', 'condition'])[
                ['subject', 'condition', 'mean_rmse', 'mean_r2', 'bad_category', 'fail_r2', 'fail_rmse']
            ].reset_index(drop=True)
        )

    bundle['id_summary'] = id_summary
    bundle['id_flagged'] = id_flagged
    bundle['ID_BAD_TRIALS'] = ID_BAD_TRIALS
    bundle['both_bad'] = both_bad
    bundle['id_only_bad'] = id_only_bad
    bundle['ik_only_bad'] = ik_only_bad
    return bundle


for joint in list(JOINT_BUNDLES):
    JOINT_BUNDLES[joint] = _annotate_id_qc(JOINT_BUNDLES[joint])


In [ ]:
# Interactive explorer: pick joint + trial
joint_dd = widgets.Dropdown(options=available_joints, value=available_joints[0], description='Joint:')
trial_dd = widgets.Dropdown(description='Trial:')
unit_dd = widgets.Dropdown(options=['N·m/kg', 'N·m'], value='N·m/kg', description='ID unit:')
time_slider = widgets.FloatRangeSlider(
    description='Time (s):', continuous_update=False, layout=widgets.Layout(width='700px'),
)
sync_out = widgets.Output()
out = widgets.Output()
ik_out = widgets.Output()


def _active_bundle():
    return JOINT_BUNDLES[joint_dd.value]


def _trial_label(bundle, trial_key: str) -> str:
    tags = []
    if trial_key in bundle['ID_BAD_TRIALS']:
        tags.append('ID')
    if trial_key in bundle['IK_BAD_TRIALS']:
        tags.append('IK')
    if tags:
        return f'{trial_key} [{"+".join(tags)} bad]'
    return trial_key


def _trial_time_rel(bundle, trial_key):
    t_id = np.asarray(bundle['TRIAL_DATA'][trial_key]['t'], dtype=np.float64)
    t_rel = t_id - t_id[0]
    if trial_key in bundle['IK_TRIAL_DATA']:
        t_ik = np.asarray(bundle['IK_TRIAL_DATA'][trial_key]['t'], dtype=np.float64)
        t_ik_rel = t_ik - t_ik[0]
        n = min(len(t_rel), len(t_ik_rel))
        t_rel = t_rel[:n]
    return t_rel


def _set_slider(bundle, trial_key):
    t_rel = _trial_time_rel(bundle, trial_key)
    tmin, tmax = float(t_rel[0]), float(t_rel[-1])
    step = max((tmax - tmin) / 500, 1e-3)
    with time_slider.hold_sync():
        time_slider.min = tmin
        time_slider.max = tmax
        time_slider.step = step
        time_slider.value = (tmin, tmax)


def _qc_badge(bundle, trial_key: str) -> str:
    if trial_key in bundle['both_bad']:
        return ' | QC: ID+IK bad'
    if trial_key in bundle['id_only_bad']:
        return ' | QC: ID bad, IK OK'
    if trial_key in bundle['ik_only_bad']:
        return ' | QC: IK bad, ID OK'
    return ''


def _refresh_trial_options(*_):
    bundle = _active_bundle()
    opts = [(_trial_label(bundle, k), k) for k in sorted(bundle['TRIAL_DATA'])]
    trial_dd.options = opts
    if opts:
        trial_dd.value = opts[0][1]


def _draw_sync(bundle, trial_key):
    d = bundle['TRIAL_DATA'][trial_key]
    names = bundle['display_names']
    n_ch = len(names)
    PREVIEW_FRAMES = bundle['preview_frames']
    if 'sync_debug' not in d:
        with sync_out:
            clear_output(wait=True)
            print('No sync preview in cache — re-run process_awinda_per_joint.ipynb.')
        return

    sd = d['sync_debug']
    fs_hz = sd['fs_hz']
    preview_n = min(PREVIEW_FRAMES, sd['n_sync'], len(d['pred_nmpkg']), len(d['id_nmpkg']))
    preview_t = np.arange(preview_n) / fs_hz

    pred_full = sd['pred_pre']
    id_full = sd['id_pre']
    n_pre_pred = min(PREVIEW_FRAMES, len(pred_full))
    n_pre_id = min(PREVIEW_FRAMES, len(id_full))
    t_pre_pred = sd['t_imu'][:n_pre_pred]
    t_pre_id = sd['t_id'][:n_pre_id] - sd['t_id'][0]

    pred_nmpkg_f = d['pred_nmpkg']
    id_nmpkg_f = d['id_nmpkg']

    fig, axs = plt.subplots(n_ch, 2, figsize=(14, 3.2 * n_ch), sharex='col')
    if n_ch == 1:
        axs = np.asarray([axs])
    for c, name in enumerate(names):
        ax_pre, ax_post = axs[c]
        ax_pre.plot(t_pre_id, id_full[:n_pre_id, c], label='OpenSim ID', color='#1e88e5', lw=1.8)
        ax_pre.plot(t_pre_pred, pred_full[:n_pre_pred, c], label='Model', color='#e53935', lw=1.4, ls='--')
        ax_pre.set_ylabel('N·m/kg')
        ax_pre.set_title(f'{name} | pre-sync')
        ax_pre.set_xlim(0, PREVIEW_FRAMES / fs_hz)
        ax_pre.axhline(0, color='gray', lw=0.5, ls=':')
        if c == 0:
            ax_pre.legend(loc='upper right', fontsize=8)

        ax_post.plot(preview_t, id_nmpkg_f[:preview_n, c], label='OpenSim ID', color='#1e88e5', lw=1.8)
        ax_post.plot(preview_t, pred_nmpkg_f[:preview_n, c], label='Model', color='#e53935', lw=1.4, ls='--')
        ax_post.set_title(f'{name} | post-sync')
        ax_post.set_xlim(0, PREVIEW_FRAMES / fs_hz)
        ax_post.axhline(0, color='gray', lw=0.5, ls=':')
        if c == 0:
            ax_post.legend(loc='upper right', fontsize=8)

    axs[-1, 0].set_xlabel('Time (s)')
    axs[-1, 1].set_xlabel('Time since sync (s)')
    fig.suptitle(
        f"{bundle['joint']} | {trial_key}{_qc_badge(bundle, trial_key)} | "
        f"lag={sd['lag']:+d} samples ({sd['lag'] / fs_hz:+.3f} s)",
        fontsize=12,
    )
    fig.tight_layout()

    fig2, ax = plt.subplots(figsize=(12, 4))
    t_imu = sd['t_imu'] - sd['t_imu'][0]
    t_id_rel = sd['t_id'] - sd['t_id'][0]
    ax.plot(t_id_rel, sd['vicon_hip_deg'], label='Vicon hip R', color='#1e88e5', lw=1.8)
    ax.plot(t_imu, sd['awinda_hip_deg'], label='Awinda hip R', color='#43a047', lw=1.4)
    lag = int(sd['lag'])
    sp = max(lag, 0)
    n_shift = max(0, min(len(t_id_rel), len(sd['awinda_hip_deg']) - sp))
    if n_shift > 0:
        ax.plot(
            t_id_rel[:n_shift], sd['awinda_hip_deg'][sp:sp + n_shift],
            label=f'Awinda shifted ({lag / fs_hz:+.2f}s)', color='#1e88e5', lw=1.2, ls='--', alpha=0.85,
        )
    ax.set_ylabel('Hip flexion R (deg)')
    ax.set_title(f"{trial_key} | angle xcorr sync (score={sd.get('xcorr_score', np.nan):.3f})")
    ax.set_xlim(0, max(PREVIEW_FRAMES / fs_hz, abs(lag) / fs_hz + 2.0))
    ax.set_xlabel('Time (s)')
    ax.legend(loc='upper right', fontsize=8)
    fig2.tight_layout()

    with sync_out:
        clear_output(wait=True)
        plt.show()
        plt.show()


def _draw_id(bundle, trial_key, unit, t_window):
    d = bundle['TRIAL_DATA'][trial_key]
    names = bundle['display_names']
    n_ch = len(names)
    t = _trial_time_rel(bundle, trial_key)
    scale = 1.0 if unit == 'N·m/kg' else d['mass_kg']
    y_pred = np.asarray(d['pred_nmpkg'], dtype=np.float64) * scale
    y_id = np.asarray(d['id_nmpkg'], dtype=np.float64) * scale
    n = min(len(t), len(y_pred), len(y_id))
    t, y_pred, y_id = t[:n], y_pred[:n], y_id[:n]
    t0, t1 = t_window

    fig, axs = plt.subplots(1, n_ch, figsize=(7 * n_ch, 4), sharey=True)
    if n_ch == 1:
        axs = [axs]
    for c, ax in enumerate(axs):
        m = d['metrics'][c]
        ax.plot(t, y_id[:, c], label='OpenSim ID', color='#1e88e5', lw=1.8)
        ax.plot(t, y_pred[:, c], label='Model', color='#e53935', lw=1.4, ls='--')
        ax.set_title(f"{names[c]} | RMSE={m['rmse_nmpkg']*scale:.3f}, R2={m['r2_nmpkg']:.3f}")
        ax.set_xlim(t0, t1)
        ax.set_ylabel(unit)
        ax.set_xlabel('Time (s)')
        ax.axhline(0, color='gray', lw=0.6, ls=':')
        if c == 0:
            ax.legend()
    fig.suptitle(
        f"{bundle['joint']} | {trial_key}{_qc_badge(bundle, trial_key)} | "
        f"lag={d['lag_samples']:+d} samples",
        fontsize=12,
    )
    fig.tight_layout()
    with out:
        clear_output(wait=True)
        plt.show()


def _draw_ik(bundle, trial_key, t_window):
    if trial_key not in bundle['IK_TRIAL_DATA']:
        with ik_out:
            clear_output(wait=True)
            print(f'No IK data for {trial_key}.')
        return

    d = bundle['IK_TRIAL_DATA'][trial_key]
    names = bundle['display_names']
    n_ch = len(names)
    y_awinda = np.asarray(d['awinda_deg'], dtype=np.float64)
    y_vicon = np.asarray(d['vicon_deg'], dtype=np.float64)
    n = min(len(y_awinda), len(y_vicon))
    t = np.asarray(d['t'][:n], dtype=np.float64)
    t = t - t[0]
    y_awinda, y_vicon = y_awinda[:n], y_vicon[:n]
    t0, t1 = t_window

    fig, axs = plt.subplots(1, n_ch, figsize=(7 * n_ch, 4), sharey=True)
    if n_ch == 1:
        axs = [axs]
    for c, ax in enumerate(axs):
        m = d['metrics'][c]
        ax.plot(t, y_vicon[:, c], label='Vicon IK', color='#1e88e5', lw=1.8)
        ax.plot(t, y_awinda[:, c], label='Awinda IK', color='#e53935', lw=1.4, ls='--')
        ax.set_title(f"{names[c]} | RMSE={m['rmse_deg']:.2f} deg, R2={m['r2']:.3f}")
        ax.set_xlim(t0, t1)
        ax.set_ylabel('Angle (deg)')
        ax.set_xlabel('Time (s)')
        ax.axhline(0, color='gray', lw=0.6, ls=':')
        if c == 0:
            ax.legend()
    fig.suptitle(
        f"{bundle['joint']} | {trial_key}{_qc_badge(bundle, trial_key)} | "
        f"Awinda IK vs Vicon IK (synced + {bundle['transient_trim_sec']:.0f}s trim)",
        fontsize=12,
    )
    fig.tight_layout()
    with ik_out:
        clear_output(wait=True)
        plt.show()


def _redraw(*_):
    bundle = _active_bundle()
    trial_key = trial_dd.value
    if not trial_key:
        return
    _draw_sync(bundle, trial_key)
    _draw_id(bundle, trial_key, unit_dd.value, time_slider.value)
    _draw_ik(bundle, trial_key, time_slider.value)


def _on_joint(change):
    _refresh_trial_options()
    if trial_dd.value:
        _set_slider(_active_bundle(), trial_dd.value)
    _redraw()


def _on_trial(change):
    _set_slider(_active_bundle(), change['new'])
    _redraw()


joint_dd.observe(_on_joint, names='value')
trial_dd.observe(_on_trial, names='value')
unit_dd.observe(_redraw, names='value')
time_slider.observe(_redraw, names='value')
_refresh_trial_options()
if trial_dd.value:
    _set_slider(_active_bundle(), trial_dd.value)
display(widgets.VBox([
    widgets.HBox([joint_dd, trial_dd, unit_dd]),
    time_slider,
    widgets.HTML('<b>Sync diagnostics</b>'),
    sync_out,
    widgets.HTML('<b>Synced ID explorer (Model vs OpenSim)</b>'),
    out,
    widgets.HTML('<b>Synced IK explorer (Awinda vs Vicon)</b>'),
    ik_out,
]))
_redraw()


## Paper-ready tables & figures

Run **after** the cache-load / IK / ID-QC cells.

Exports to `analysis/paper_outputs/awinda_per_joint/{joint}/` for each available joint.


In [ ]:
PAPER_EXCLUDE_SUBJECTS = {'AB02_Oscar'}
PAPER_EXCLUDE_TRIALS = {'AB05_Maria::LG_0p8mps'}

PAPER_RC = {
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
}
PALETTE = {'ID': '#1e88e5', 'IK': '#e53935', 'RMSE': '#3949ab', 'R2': '#43a047'}


def _paper_exclude_trial(trial_key: str) -> bool:
    subject, _ = trial_key.split('::', 1)
    return subject in PAPER_EXCLUDE_SUBJECTS or trial_key in PAPER_EXCLUDE_TRIALS


def _filter_paper_trials(df, trial_col='trial'):
    keep = ~df[trial_col].map(_paper_exclude_trial)
    return df.loc[keep].copy()


def _parse_condition(condition: str):
    task, speed = condition.split('_', 1)
    speed_mps = float(speed.replace('mps', '').replace('p', '.'))
    return task, speed_mps


def _mean_std(series, decimals=3):
    x = np.asarray(series, dtype=np.float64)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return '—'
    return f'{np.mean(x):.{decimals}f} ± {np.std(x, ddof=1):.{decimals}f}'


def _attach_trial_meta(df, trial_col='trial'):
    df = df.copy()
    meta = df[trial_col].str.split('::', n=1, expand=True)
    df['subject'] = meta[0]
    df['condition'] = meta[1]
    parsed = df['condition'].apply(_parse_condition)
    df['task'] = [p[0] for p in parsed]
    df['speed_mps'] = [p[1] for p in parsed]
    return df


def _save_table(df, out_dir: Path, stem: str, caption=''):
    csv_path = out_dir / f'{stem}.csv'
    tex_path = out_dir / f'{stem}.tex'
    df.to_csv(csv_path, index=False)
    tex = df.to_latex(index=False, escape=True, caption=caption, label=f'tab:{stem}')
    tex_path.write_text(tex)
    return csv_path, tex_path


def export_paper_for_joint(bundle: dict):
    joint = bundle['joint']
    TRIAL_DATA = bundle['TRIAL_DATA']
    detail_df = bundle['detail_df']
    summary_df = bundle['summary_df']
    ik_detail_df = bundle['ik_detail_df']
    ik_summary_df = bundle['ik_summary_df']
    channels = bundle['channels']
    display_names = bundle['display_names']

    if detail_df.empty or ik_detail_df.empty:
        print(f'[{joint}] skip paper export — empty metrics')
        return

    out_dir = PROJECT_ROOT / 'analysis' / 'paper_outputs' / 'awinda_per_joint' / joint
    out_dir.mkdir(parents=True, exist_ok=True)

    paper_keys = sorted(k for k in TRIAL_DATA if not _paper_exclude_trial(k))
    excluded = sorted(k for k in TRIAL_DATA if _paper_exclude_trial(k))
    print(f'[{joint}] Paper subset: {len(paper_keys)} trials (excluded {len(excluded)})')

    detail_df_paper = _filter_paper_trials(detail_df)
    ik_detail_df_paper = _filter_paper_trials(ik_detail_df)
    summary_df_paper = _filter_paper_trials(summary_df)
    ik_summary_df_paper = _filter_paper_trials(ik_summary_df)

    id_detail = _attach_trial_meta(detail_df_paper)
    ik_detail = _attach_trial_meta(ik_detail_df_paper)

    rows = []
    for ch, lab in zip(channels, display_names):
        sub = id_detail[id_detail['channel'] == ch]
        rows.append({
            'joint': lab,
            'channel': ch,
            'n_trials': sub['trial'].nunique(),
            'rmse_nmpkg_mean_std': _mean_std(sub['rmse_nmpkg']),
            'r2_nmpkg_mean_std': _mean_std(sub['r2_nmpkg']),
            'rmse_nmpkg_mean': float(sub['rmse_nmpkg'].mean()),
            'rmse_nmpkg_std': float(sub['rmse_nmpkg'].std(ddof=1)),
            'r2_nmpkg_mean': float(sub['r2_nmpkg'].mean()),
            'r2_nmpkg_std': float(sub['r2_nmpkg'].std(ddof=1)),
        })
    table_id = pd.DataFrame(rows)

    rows = []
    for ch, lab in zip(channels, display_names):
        sub = ik_detail[ik_detail['channel'] == ch]
        rows.append({
            'joint': lab,
            'channel': ch,
            'n_trials': sub['trial'].nunique(),
            'rmse_deg_mean_std': _mean_std(sub['rmse_deg']),
            'r2_mean_std': _mean_std(sub['r2']),
            'rmse_deg_mean': float(sub['rmse_deg'].mean()),
            'rmse_deg_std': float(sub['rmse_deg'].std(ddof=1)),
            'r2_mean': float(sub['r2'].mean()),
            'r2_std': float(sub['r2'].std(ddof=1)),
        })
    table_ik = pd.DataFrame(rows)

    id_trial = _attach_trial_meta(summary_df_paper.copy())
    id_trial['lag_s'] = id_trial['trial'].map(lambda k: TRIAL_DATA[k]['lag_seconds'])
    id_trial = id_trial.rename(columns={
        'mean_rmse': 'mean_rmse_nmpkg', 'mean_r2': 'mean_r2_id', 'n': 'n_samples',
    })

    ik_trial = _attach_trial_meta(ik_summary_df_paper.copy())
    ik_trial = ik_trial.rename(columns={'mean_r2': 'mean_r2_ik', 'n': 'n_samples'})

    overall_rows = pd.DataFrame([
        {
            'metric': f'ID ({joint} model vs OpenSim)',
            'unit': 'N·m/kg / R2',
            'rmse_mean_std': _mean_std(id_detail['rmse_nmpkg']),
            'r2_mean_std': _mean_std(id_detail['r2_nmpkg']),
            'n_observations': len(id_detail),
        },
        {
            'metric': f'IK ({joint} Awinda vs Vicon)',
            'unit': 'deg / R2',
            'rmse_mean_std': _mean_std(ik_detail['rmse_deg']),
            'r2_mean_std': _mean_std(ik_detail['r2']),
            'n_observations': len(ik_detail),
        },
    ])

    for stem, df, cap in [
        ('table1_id_per_side', table_id[['joint', 'n_trials', 'rmse_nmpkg_mean_std', 'r2_nmpkg_mean_std']],
         f'{joint} model ID vs OpenSim (R/L).'),
        ('table2_ik_per_side', table_ik[['joint', 'n_trials', 'rmse_deg_mean_std', 'r2_mean_std']],
         f'{joint} Awinda vs Vicon IK (R/L).'),
        ('table3_id_per_trial', id_trial.drop(columns=['trial'], errors='ignore').round(4),
         f'{joint} per-trial ID summary.'),
        ('table4_ik_per_trial', ik_trial.drop(columns=['trial'], errors='ignore').round(4),
         f'{joint} per-trial IK summary.'),
        ('table5_overall_summary', overall_rows, f'{joint} overall summary.'),
    ]:
        _save_table(df, out_dir, stem, caption=cap)

    x = np.arange(len(channels))
    labels = display_names
    with plt.rc_context(PAPER_RC):
        fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.2), sharex=True)
        axes[0].bar(x, table_id['rmse_nmpkg_mean'], yerr=table_id['rmse_nmpkg_std'],
                    color=PALETTE['RMSE'], capsize=3, alpha=0.9)
        axes[0].set_ylabel('RMSE (N·m/kg)')
        axes[0].set_title('(a) ID RMSE')
        axes[1].bar(x, table_id['r2_nmpkg_mean'], yerr=table_id['r2_nmpkg_std'],
                    color=PALETTE['R2'], capsize=3, alpha=0.9)
        axes[1].set_ylabel('R2')
        axes[1].set_ylim(0, 1.05)
        axes[1].set_title('(b) ID R2')
        for ax in axes:
            ax.set_xticks(x)
            ax.set_xticklabels(labels)
        fig.suptitle(f'{joint.title()} model ID vs OpenSim (n={len(paper_keys)} trials)', y=1.02)
        fig.tight_layout()
        fig.savefig(out_dir / 'fig1_id_per_side.pdf')
        fig.savefig(out_dir / 'fig1_id_per_side.png')
        plt.show()

        fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.2), sharex=True)
        axes[0].bar(x, table_ik['rmse_deg_mean'], yerr=table_ik['rmse_deg_std'],
                    color=PALETTE['IK'], capsize=3, alpha=0.85)
        axes[0].set_ylabel('RMSE (deg)')
        axes[0].set_title('(a) IK RMSE')
        axes[1].bar(x, table_ik['r2_mean'], yerr=table_ik['r2_std'],
                    color=PALETTE['R2'], capsize=3, alpha=0.85)
        axes[1].set_ylabel('R2')
        axes[1].set_ylim(0, 1.05)
        axes[1].set_title('(b) IK R2')
        for ax in axes:
            ax.set_xticks(x)
            ax.set_xticklabels(labels)
        fig.suptitle(f'{joint.title()} Awinda IK vs Vicon (n={len(paper_keys)} trials)', y=1.02)
        fig.tight_layout()
        fig.savefig(out_dir / 'fig2_ik_per_side.pdf')
        fig.savefig(out_dir / 'fig2_ik_per_side.png')
        plt.show()

    print(f'[{joint}] Paper outputs -> {out_dir}')
    display(table_id[['joint', 'n_trials', 'rmse_nmpkg_mean_std', 'r2_nmpkg_mean_std']])
    display(table_ik[['joint', 'n_trials', 'rmse_deg_mean_std', 'r2_mean_std']])
    display(overall_rows)


for joint in available_joints:
    export_paper_for_joint(JOINT_BUNDLES[joint])
